<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/selic/Selic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Selic — meta (nível) e over (acumulado em 12 meses)

Motor de coleta e publicação das **duas** séries da família `selic`. Elas medem coisas
diferentes e por isso têm `medida` diferente:

| série | fonte (SGS) | o que é | `medida` |
|---|---|---|---|
| `selic-meta` | **432** — meta Selic do Copom, % a.a. | **nível** da meta no mês, anualizado: a taxa que o Copom define | `nivel` |
| `selic-over-acum-12m` | **11** — Selic efetiva, % a.d. | taxa efetiva **acumulada** nos 12 meses de calendário até o mês | `acumulado-12m` |

- **Endpoints:**
  `https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json` ·
  `https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados?formato=json`
- **Unidade publicada:** `percentual` (`%`, 2 casas), `eixo: percentual`, `granularidade: mensal`.
- **Arquivos:** `indicadores/selic/selic-meta.json` e `indicadores/selic/selic-over-acum-12m.json`.

## Por que a meta não é acumulada

A meta é uma taxa **anualizada** — `15%` significa `15% ao ano`. Acumulá-la em 12 meses não é
uma leitura da meta: exige converter cada meta mensal em taxa mensal equivalente,
`(1 + m/100)^(1/12)`, e capitalizar 12 dessas — um construto que vale exatamente a meta quando
ela fica parada o ano inteiro e fica defasado em até 12 meses quando o Copom se move. Nenhuma
série oficial faz isso, e ninguém diz "Selic a 15%" querendo dizer o acumulado.

Quem rende e, portanto, se acumula é a **Selic efetiva (over)**. É ela que vira
`selic-over-acum-12m`, composta dia a dia sobre os fatores publicados.


## Coleta

Leitura integral a cada execução — nada de atualização incremental, nada de ler o JSON já
publicado para decidir o período. O SGS limita séries diárias a janelas de 10 anos por
requisição, então a coleta é fatiada em décadas e remontada por data.

In [1]:
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()

def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name
PASTA = RAIZ / "indicadores" / "selic" if (RAIZ / "indicadores" / "selic").is_dir() else RAIZ
DESTINO = Path(os.environ.get("SELIC_SAIDA", PASTA))
ARQUIVO_OVER = DESTINO / "selic-over-acum-12m.json"
ARQUIVO_META = DESTINO / "selic-meta.json"

URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
SERIE_OVER = 11
SERIE_META = 432
SERIE_CONFERENCIA = 4390
URL_OVER = URL.format(serie=SERIE_OVER) + "?formato=json"
URL_META = URL.format(serie=SERIE_META) + "?formato=json"
# As séries 11 e 432 são diárias: o endpoint sem janela devolve 406. O urlFonte
# publicado aponta para a página estável do BCB, que abre e permite auditar o número.
URL_OVER_FONTE = "https://www.bcb.gov.br/estatisticas/txjuros"
URL_META_FONTE = "https://www.bcb.gov.br/controleinflacao/historicotaxasjuros"

# Recorte editorial: mesma janela das demais famílias (IPCA e IGP-M começam em 1995-01).
# A SGS 11 tem dados desde 1986-06; o acumulado de 12 meses começa 11 meses depois do
# início da coleta, em 1995-12.
INICIO_OVER = date(1995, 1, 1)
HOJE = date.today()
PRIMEIRO_DIA_MES_CORRENTE = HOJE.replace(day=1)


def baixar(serie, inicio, tentativas=4):
    # Baixa uma série SGS desde `inicio`, em janelas de 10 anos (limite do SGS para
    # séries diárias), com nova tentativa e espera crescente quando o serviço falha.
    # Devolve [(date, float)] ascendente, sem data repetida.
    leituras = {}
    cursor = inicio
    while cursor <= HOJE:
        fim = min(HOJE, date(cursor.year + 9, 12, 31))
        parametros = {
            "formato": "json",
            "dataInicial": cursor.strftime("%d/%m/%Y"),
            "dataFinal": fim.strftime("%d/%m/%Y"),
        }
        resposta = None
        for tentativa in range(tentativas):
            try:
                resposta = requests.get(
                    URL.format(serie=serie), params=parametros, timeout=60
                )
                if resposta.status_code == 404:
                    break
                resposta.raise_for_status()
                for item in resposta.json():
                    dia, mes, ano = (int(parte) for parte in item["data"].split("/"))
                    leituras[date(ano, mes, dia)] = float(item["valor"])
                break
            except (requests.RequestException, ValueError):
                if tentativa == tentativas - 1:
                    raise
                time.sleep(2 ** tentativa)
        cursor = date(fim.year + 1, 1, 1)
    if not leituras:
        raise ValueError(f"a série SGS {serie} não retornou dados")
    return sorted(leituras.items())


def por_mes(leituras):
    # Agrupa leituras diárias por mês de calendário, em ordem.
    meses = {}
    for dia, valor in leituras:
        meses.setdefault((dia.year, dia.month), []).append(valor)
    return dict(sorted(meses.items()))


def exigir_meses_continuos(meses, rotulo):
    # Falha se faltar um mês no meio da série: lacuna não pode virar janela torta.
    chaves = list(meses)
    for (ano_a, mes_a), (ano_b, mes_b) in zip(chaves, chaves[1:]):
        if (ano_b * 12 + mes_b) - (ano_a * 12 + mes_a) != 1:
            raise ValueError(
                f"lacuna de mês em {rotulo}: {ano_a}-{mes_a:02d} -> {ano_b}-{mes_b:02d}"
            )


bruto_over = [leitura for leitura in baixar(SERIE_OVER, INICIO_OVER) if leitura[0] >= INICIO_OVER]
bruto_meta = baixar(SERIE_META, date(1999, 1, 1))
conferencia_4390 = [
    leitura for leitura in baixar(SERIE_CONFERENCIA, INICIO_OVER) if leitura[0] >= INICIO_OVER
]

print(f"SGS {SERIE_OVER}: {len(bruto_over)} leituras diárias "
      f"({bruto_over[0][0]:%Y-%m-%d} a {bruto_over[-1][0]:%Y-%m-%d})")
print(f"SGS {SERIE_META}: {len(bruto_meta)} leituras diárias "
      f"({bruto_meta[0][0]:%Y-%m-%d} a {bruto_meta[-1][0]:%Y-%m-%d})")
print(f"SGS {SERIE_CONFERENCIA} (conferência mensal): {len(conferencia_4390)} leituras")
print(f"Pasta de saída: {relativo(DESTINO)}")

SGS 11: 7992 leituras diárias (1995-01-02 a 2026-11-04)
SGS 432: 10108 leituras diárias (1999-03-05 a 2026-11-05)
SGS 4390 (conferência mensal): 383 leituras
Pasta de saída: indicadores/selic


## Selic over — composição em janela de 12 meses de calendário

Cada mês de calendário vira um fator `F_m = ∏(1 + i_d/100)` sobre **todos** os dias
publicados naquele mês. O acumulado de 12 meses é o produto dos 12 fatores mensais
consecutivos, menos 1. É a mesma conta que o mercado faz a partir da Selic acumulada no mês
(SGS 4390), só que aqui com a precisão cheia da série diária.

Meses ainda abertos não são publicados: a janela de 12 meses só fecha no último dia do mês.

In [2]:
meses_over = por_mes(bruto_over)
exigir_meses_continuos(meses_over, f"SGS {SERIE_OVER}")

fatores_over = {
    chave: math.prod(1 + valor / 100 for valor in valores)
    for chave, valores in meses_over.items()
}


def acumulado_mensal(fatores, casas=12):
    # Produto dos `casas` fatores mensais consecutivos, menos 1, datado no mês de fechamento.
    itens = list(fatores.items())
    pontos = []
    for posicao in range(casas - 1, len(itens)):
        janela = itens[posicao - casas + 1 : posicao + 1]
        fator = math.prod(valor for _, valor in janela)
        (ano, mes), _ = janela[-1]
        pontos.append((date(ano, mes, 1), (fator - 1) * 100))
    return pontos


def arredondar(valor):
    # 2 casas com meio para cima (convenção comercial).
    return float(Decimal(repr(float(valor))).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


bruto_over_12m = acumulado_mensal(fatores_over)
over_12m = [(dia, valor) for dia, valor in bruto_over_12m if dia < PRIMEIRO_DIA_MES_CORRENTE]

if not over_12m:
    raise ValueError("nenhum mês fechado no período publicado")

print(f"Janelas completas: {len(bruto_over_12m)} "
      f"({bruto_over_12m[0][0]:%Y-%m} a {bruto_over_12m[-1][0]:%Y-%m})")
print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Publicado: {len(over_12m)} meses, {over_12m[0][0]:%Y-%m} a {over_12m[-1][0]:%Y-%m}")
print(f"Primeiro: {over_12m[0][0]:%Y-%m} = {over_12m[0][1]:.4f}%")
print(f"Último:   {over_12m[-1][0]:%Y-%m} = {over_12m[-1][1]:.4f}%")

# Conferência 1: o fator mensal da série diária tem de reproduzir a Selic acumulada no mês
# que o BCB publica na SGS 4390 (arredondada a 2 casas).
oficial_4390 = {
    (dia.year, dia.month): valor for dia, valor in conferencia_4390
}
divergencias = [
    (chave, (fator - 1) * 100, oficial_4390[chave])
    for chave, fator in fatores_over.items()
    if chave in oficial_4390 and abs((fator - 1) * 100 - oficial_4390[chave]) > 0.0051
]
print(f"\nFator mensal x SGS {SERIE_CONFERENCIA}: {len(divergencias)} divergências acima de 0,005 p.p. "
      f"em {len(set(fatores_over) & set(oficial_4390))} meses comuns")
assert not divergencias, divergencias[:5]

# Conferência 2: o acumulado também fecha quando composto só da série oficial 4390.
ultimo = over_12m[-1][0]
chaves_janela = [
    (ultimo.year + (ultimo.month - 1 - deslocamento) // 12,
     (ultimo.month - 1 - deslocamento) % 12 + 1)
    for deslocamento in range(11, -1, -1)
]
fator_oficial = math.prod(1 + oficial_4390[chave] / 100 for chave in chaves_janela)
print(f"Última janela pela SGS {SERIE_CONFERENCIA}: {(fator_oficial - 1) * 100:.4f}% "
      f"x {over_12m[-1][1]:.4f}% pela SGS {SERIE_OVER}")
assert abs((fator_oficial - 1) * 100 - over_12m[-1][1]) < 0.02

Janelas completas: 372 (1995-12 a 2026-11)
Mês corrente excluído: 2026-11
Publicado: 371 meses, 1995-12 a 2026-10
Primeiro: 1995-12 = 53.0872%
Último:   2026-10 = 14.2438%

Fator mensal x SGS 4390: 0 divergências acima de 0,005 p.p. em 383 meses comuns
Última janela pela SGS 4390: 14.2315% x 14.2438% pela SGS 11


## Selic meta — nível do mês (sem acumular)

A meta é a taxa que o Copom define, expressa ao ano. O valor publicado para cada mês é a
própria meta em vigor no **último dia do mês** — anualizada, como o BCB a divulga. Nada de
acumular: acumular a meta seria inventar uma taxa de retorno que ela não é (ver a nota no
topo). Quando o Copom muda a meta no meio do mês, o ponto publica o nível novo, o que estava
em vigor no fechamento; não há média das leituras diárias.

O mês corrente é descartado (o Copom pode se reunir no meio dele). Março/1999 entra com os dias
disponíveis: a série começa em 05/03 com 45%, e o mês fecha em 42%.


In [3]:
meses_meta = por_mes(bruto_meta)
exigir_meses_continuos(meses_meta, f"SGS {SERIE_META}")

primeiro_mes_meta = next(iter(meses_meta))
if primeiro_mes_meta != (1999, 3):
    raise ValueError(f"primeiro mês inesperado na SGS {SERIE_META}: {primeiro_mes_meta}")
print(f"Primeiro mês da fonte: {primeiro_mes_meta[0]}-{primeiro_mes_meta[1]:02d} "
      f"({len(meses_meta[primeiro_mes_meta])} dias; a série começa em 05/03/1999)")

# A 432 publica um valor por dia de calendário (inclusive fim de semana). A meta do mês é o
# nível em vigor no último dia do mês: a última leitura publicada. Quando o Copom muda a meta
# no meio do mês, o ponto já publica o nível novo, sem média das leituras diárias.
meta_nivel = [
    (date(ano, mes, 1), valores[-1])
    for (ano, mes), valores in meses_meta.items()
    if date(ano, mes, 1) < PRIMEIRO_DIA_MES_CORRENTE
]

if not meta_nivel:
    raise ValueError("nenhum mês fechado no período publicado")

print(f"Publicado: {len(meta_nivel)} meses, {meta_nivel[0][0]:%Y-%m} a {meta_nivel[-1][0]:%Y-%m}")
print(f"Primeiro: {meta_nivel[0][0]:%Y-%m} = {meta_nivel[0][1]:.4f}% ao ano")
print(f"Último:   {meta_nivel[-1][0]:%Y-%m} = {meta_nivel[-1][1]:.4f}% ao ano")

# A meta é o nível: não há janela de 12 meses. jul/2016 estava em 14,25% ao ano.
alvo = date(2016, 7, 1)
assert abs(dict(meta_nivel)[alvo] - 14.25) < 1e-9, dict(meta_nivel)[alvo]
assert abs(dict(meta_nivel)[date(2025, 10, 1)] - 15.00) < 1e-9
print(f"Controle: {alvo:%Y-%m} = {dict(meta_nivel)[alvo]:.4f}% ao ano (meta em vigor)")


Primeiro mês da fonte: 1999-03 (27 dias; a série começa em 05/03/1999)
Publicado: 332 meses, 1999-03 a 2026-10
Primeiro: 1999-03 = 42.0000% ao ano
Último:   2026-10 = 13.7500% ao ano
Controle: 2016-07 = 14.2500% ao ano (meta em vigor)


## Escrita dos JSON do contrato

Cada arquivo é gravado inteiro — `versao`, `id`, `unidade`, `eixo`, `medida`,
`granularidade`, `fonte`, `urlFonte` e `dados` — sem ler nem preservar nada do que está
publicado. Pontos no formato `{"data": "YYYY-MM-DD", "valor": ...}`, arredondados a 2 casas.

In [4]:
def montar_serie(id_serie, fonte_url, medida, pontos, fonte):
    return {
        "versao": 1,
        "id": id_serie,
        "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 2},
        "eixo": "percentual",
        "medida": medida,
        "granularidade": "mensal",
        "fonte": fonte,
        "urlFonte": fonte_url,
        "dados": [
            {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)}
            for dia, valor in pontos
        ],
    }


serie_over = montar_serie("selic-over-acum-12m", URL_OVER_FONTE, "acumulado-12m", over_12m, "Banco Central do Brasil (SGS 11); acumulado 12m calculado pelo site")
serie_meta = montar_serie("selic-meta", URL_META_FONTE, "nivel", meta_nivel, "Banco Central do Brasil")

DESTINO.mkdir(parents=True, exist_ok=True)
for arquivo, serie in ((ARQUIVO_OVER, serie_over), (ARQUIVO_META, serie_meta)):
    with arquivo.open("w", encoding="utf-8") as destino:
        json.dump(serie, destino, ensure_ascii=False, indent=2)
        destino.write("\n")
    print(f"{len(serie['dados'])} pontos -> {relativo(arquivo)}")
    print(f"  {serie['dados'][0]}")
    print(f"  {serie['dados'][-1]}")


371 pontos -> indicadores/selic/selic-over-acum-12m.json
  {'data': '1995-12-01', 'valor': 53.09}
  {'data': '2026-10-01', 'valor': 14.24}
332 pontos -> indicadores/selic/selic-meta.json
  {'data': '1999-03-01', 'valor': 42.0}
  {'data': '2026-10-01', 'valor': 13.75}


## Validação inline

A célula falha se o contrato, a ordem, a unicidade e a continuidade dos meses, a finitude dos
valores, a composição ou a conferência contra a fonte forem violados. Os números citados em
`metodologia.md` também são conferidos aqui como âncoras de regressão; a verificação externa
está registrada em `metodologia.md`, não nesta célula.

In [5]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
ESPERADO = {
    # id: (urlFonte, medida, primeiro ponto)
    "selic-over-acum-12m": (URL_OVER_FONTE, "acumulado-12m", "1995-12-01"),
    "selic-meta": (URL_META_FONTE, "nivel", "1999-03-01"),
}


def validar(serie, arquivo):
    assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
    assert serie["versao"] == 1, serie["versao"]
    assert serie["id"] in ESPERADO, serie["id"]
    assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 2}, serie["unidade"]
    assert serie["eixo"] == "percentual", serie["eixo"]
    assert serie["granularidade"] == "mensal", serie["granularidade"]
    assert serie["fonte"] == {
        "selic-over-acum-12m": "Banco Central do Brasil (SGS 11); acumulado 12m calculado pelo site",
        "selic-meta": "Banco Central do Brasil",
    }[serie["id"]], serie["fonte"]
    url_esperada, medida_esperada, inicio_esperado = ESPERADO[serie["id"]]
    assert serie["medida"] == medida_esperada, serie["medida"]
    assert serie["urlFonte"] == url_esperada, serie["urlFonte"]

    dados = serie["dados"]
    assert dados, "série vazia"
    for ponto in dados:
        assert set(ponto) == {"data", "valor"}, sorted(ponto)
        assert isinstance(ponto["data"], str), ponto
        assert not isinstance(ponto["valor"], bool), ponto
        assert isinstance(ponto["valor"], (int, float)), ponto
        assert math.isfinite(ponto["valor"]), ponto

    dias = [date.fromisoformat(ponto["data"]) for ponto in dados]
    assert all(a < b for a, b in zip(dias, dias[1:])), "datas fora de ordem ou repetidas"
    assert all(dia.day == 1 for dia in dias), "dia diferente de 01"
    assert all(
        (b.year * 12 + b.month) - (a.year * 12 + a.month) == 1 for a, b in zip(dias, dias[1:])
    ), "mês faltando ou repetido"
    assert dados[0]["data"] == inicio_esperado, dados[0]["data"]
    assert dias[-1] < PRIMEIRO_DIA_MES_CORRENTE, dados[-1]["data"]

    with arquivo.open(encoding="utf-8") as origem:
        assert json.load(origem) == serie, "arquivo gravado difere do objeto validado"
    print(f"ok {serie['id']}: {len(dados)} meses, {dados[0]['data']} a {dados[-1]['data']}, "
          f"último {dados[-1]['valor']:.2f}%")
    return {ponto["data"]: ponto["valor"] for ponto in dados}


por_over = validar(serie_over, ARQUIVO_OVER)
por_meta = validar(serie_meta, ARQUIVO_META)

assert len(serie_over["dados"]) == len(over_12m)
assert len(serie_meta["dados"]) == len(meta_nivel)

# Os valores gravados são o resultado recalculado acima, sem sobra nem falta.
assert serie_over["dados"] == [
    {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)} for dia, valor in over_12m
]
assert serie_meta["dados"] == [
    {"data": dia.strftime("%Y-%m-%d"), "valor": arredondar(valor)} for dia, valor in meta_nivel
]

# --- over: âncoras de regressão (não são a verificação independente) ----------------
# Os valores abaixo saem do mesmo cálculo da série; a verificação externa (Brasil
# Indicadores, 04/11/2026) está registrada em metodologia.md.
assert por_over["2025-10-01"] == 13.71, por_over["2025-10-01"]
assert por_over["2026-09-01"] == 14.47, por_over["2026-09-01"]
# Choques e datas recentes citados em metodologia.md.
assert por_over["2002-12-01"] == 19.17, por_over["2002-12-01"]
assert por_over["2008-12-01"] == 12.48, por_over["2008-12-01"]
assert por_over["2020-12-01"] == 2.76, por_over["2020-12-01"]
assert por_over["2021-12-01"] == 4.42, por_over["2021-12-01"]
assert por_over["2026-10-01"] == 14.24, por_over["2026-10-01"]

# --- meta: nível, não acumulado ------------------------------------------------------
# jul/2016: meta constante de 14,25% ao ano. Como nível, o valor é exatamente 14,25.
assert abs(por_meta["2016-07-01"] - 14.25) < 1e-9, por_meta["2016-07-01"]
assert abs(por_meta["2025-10-01"] - 15.00) < 1e-9, por_meta["2025-10-01"]
assert por_meta["2003-12-01"] == 16.50, por_meta["2003-12-01"]
# Março/1999 é o primeiro mês da fonte (a partir de 05/03); o mês fecha em 42,00.
assert por_meta["1999-03-01"] == 42.00, por_meta["1999-03-01"]
# Meses em que o Copom se moveu e a média diária divergia do nível de fim de mês.
assert por_meta["2024-12-01"] == 12.25, por_meta["2024-12-01"]
assert por_meta["2025-06-01"] == 15.00, por_meta["2025-06-01"]
assert por_meta["2026-09-01"] == 13.75, por_meta["2026-09-01"]

# A diferença em relação ao construto acumulado é o motivo de a meta ser publicada como nível:
# em dez/2003 a meta em vigor no fim do mês era 16,50% ao ano, e "acumulada em 12 meses" daria 23,03%.
def acumulado_12m_do_nivel(pontos, alvo):
    posicao = [dia for dia, _ in pontos].index(alvo)
    janela = pontos[posicao - 11 : posicao + 1]
    fator = math.prod((1 + valor / 100) ** (1 / 12) for _, valor in janela)
    return (fator - 1) * 100


construto = acumulado_12m_do_nivel(meta_nivel, date(2003, 12, 1))
print(f"Construto que não é publicado — meta 'acumulada' em 2003-12: {construto:.2f}% "
      f"x nível {por_meta['2003-12-01']:.2f}%")
assert abs(construto - 23.03) < 0.01, construto

print("\ntudo certo: over acumulado em janelas de 12 meses; meta publicada como nível")


ok selic-over-acum-12m: 371 meses, 1995-12-01 a 2026-10-01, último 14.24%
ok selic-meta: 332 meses, 1999-03-01 a 2026-10-01, último 13.75%
Construto que não é publicado — meta 'acumulada' em 2003-12: 23.03% x nível 16.50%

tudo certo: over acumulado em janelas de 12 meses; meta publicada como nível
